# Supplemental Figure S9
## Curated Gene Expression Patterns

Assembles panels from `curated_gene_patterns_pca_*.svg` with:
- **Helvetica 7pt** for general text (axis/colorbar labels)
- **Helvetica Oblique 7pt** for gene names
- **Helvetica 8pt** for gene class headers

## Step 1 — Collect genes and their categories from panel SVGs

Each `curated_gene_patterns_pca_*.svg` file in the snRNAseq figure directory
contains one or more gene expression plots for a specific functional category.
The category name and gene names are embedded as `<text>` elements in the SVG.

Below we scan those files, extract the gene names per category, and display the
mapping.

In [1]:
!pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 63.7 MB/s eta 0:00:0000:0100:01


In [1]:
%load_ext autoreload
%autoreload 2


import os, sys, glob, re
from lxml import etree

notebook_dir = os.path.dirname(os.path.abspath(''))
if notebook_dir not in sys.path:
    sys.path.append(os.path.dirname(notebook_dir))
from notebooks.config import *
SVG_NS = "http://www.w3.org/2000/svg"

# ── Discover PCA panel SVGs (skip files with broken newline-containing names) ──
svg_files = sorted(glob.glob(os.path.join(SNRNA_FIGURE_DIR, "curated_gene_patterns_pca_*.svg")))
svg_files = [f for f in svg_files if "\n" not in f]

# ── Extract gene names and category label from each SVG ──
category_genes = {}

for svg_path in svg_files:
    tree = etree.parse(svg_path)
    root = tree.getroot()

    # Collect text elements with large font-size (>=11 pt) — these are labels
    big_texts = []
    for txt_el in root.iter(f"{{{SVG_NS}}}text"):
        content = (txt_el.text or "").strip()
        if not content:
            continue
        style = txt_el.get("style", "")
        m = re.search(r"font-size:\s*([\d.]+)", style)
        font_size = float(m.group(1)) if m else 0
        if font_size >= 11:
            y = float(txt_el.get("y", "0"))
            big_texts.append((y, content))

    if not big_texts:
        continue

    # Sort by y-position: topmost label is the category, the rest are gene names
    big_texts.sort(key=lambda t: t[0])
    category = big_texts[0][1]
    genes = [t[1] for t in big_texts[1:]]

    category_genes[category] = genes

# ── Display results ──
total = 0
for cat, genes in category_genes.items():
    total += len(genes)
    print(f"{cat:35s}  ({len(genes):d})  {', '.join(genes)}")

print(f"\nTotal: {len(category_genes)} categories, {total} genes")

PROJECT_ROOT: /root/capsule/code
CODE_DIR: /root/capsule/code
CODE_DIR exists: True
Python path includes CODE_DIR: True
Axon guidance                        (7)  Epha6, Plxna2, Robo1, Epha7, Plxna4, Sema3c, Slit2
Cell adhesion                        (9)  Csmd3, Lrrtm4, Ncam2, Cdh6, Cntn4, Fat3, Pcdh7, Sdk2, Tmem132d
ECM                                  (5)  Chsy3, Col18a1, Col6a5, Hs6st3, Ndst3
Ion channels                         (4)  Hcn1, Scn9a, Kcnh1, Kctd8
Neuromodulator signaling             (4)  Trhr, Sctr, Tacr3, Olfr78
Neurotransmitter signaling           (1)  Gabrg1
Synaptic transmission                (1)  Cadps2
Transcription                        (4)  Esr1, Nrip1, Tshz1, Tshz2
Transporters                         (2)  Slc16a10, Slc39a12

Total: 9 categories, 37 genes


In [2]:
FONT_PATH

'/root/capsule/code/utils/Helvetica.ttc'

In [3]:
PROJECT_ROOT

'/root/capsule/code'

In [4]:
SNRNA_FIGURE_DIR

'/root/capsule/output/figures/snRNAseq'

In [5]:
import subprocess, sys

# Install cairosvg if not available
try:
    import cairosvg
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cairosvg", "-q"])
    import cairosvg

try:
    import lxml.etree
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lxml", "-q"])
    import lxml.etree

In [6]:
# Check the TTC font faces
from fontTools.ttLib import TTCollection
ttc = TTCollection("/root/capsule/code/utils/Helvetica.ttc")
for i, font in enumerate(ttc.fonts):
    name_table = font['name']
    family = name_table.getDebugName(1)
    subfamily = name_table.getDebugName(2)
    print(f"  Font {i}: {family} - {subfamily}")

  Font 0: Helvetica - Regular
  Font 1: Helvetica - Bold
  Font 2: Helvetica - Oblique
  Font 3: Helvetica - Bold Oblique
  Font 4: Helvetica - Light
  Font 5: Helvetica - Light Oblique


In [7]:
import os, glob, re, copy, io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.image import imread
from PIL import Image
from lxml import etree

# ── Project config ──

configure_matplotlib()

# ── Font properties ──
font_text  = fm.FontProperties(fname=FONT_PATH, size=7)           # Helvetica 7pt
font_class = fm.FontProperties(fname=FONT_PATH, size=8)           # Helvetica 8pt

# Extract and register Helvetica Oblique for gene names
oblique_path = os.path.join(CODE_DIR, 'utils', "Helvetica-Oblique.ttf")
if not os.path.exists(oblique_path):
    from fontTools.ttLib import TTCollection
    ttc = TTCollection(FONT_PATH)
    ttc.fonts[2].save(oblique_path)  # index 2 = Oblique
    print(f"Extracted Oblique to {oblique_path}")

fm.fontManager.addfont(oblique_path)
font_gene = fm.FontProperties(fname=oblique_path, size=7)         # Helvetica Oblique 7pt

print("Font text :", font_text.get_name(), font_text.get_size_in_points())
print("Font gene :", font_gene.get_name(), font_gene.get_size_in_points(), font_gene.get_style())
print("Font class:", font_class.get_name(), font_class.get_size_in_points())

Font text : Helvetica 7.0
Font gene : Helvetica 7.0 normal
Font class: Helvetica 8.0


In [8]:
FONT_PATH

'/root/capsule/code/utils/Helvetica.ttc'

In [9]:
# ── Discover SVG panels (PCA versions) ──
svg_dir = SNRNA_FIGURE_DIR
svg_files = sorted(glob.glob(os.path.join(svg_dir, "curated_gene_patterns_pca_*.svg")))

# Filter out files with broken names (newlines etc.)
svg_files = [f for f in svg_files if "\n" not in f and os.path.basename(f).count("_") >= 4]

print(f"Found {len(svg_files)} PCA panels:")
for f in svg_files:
    print(f"  {os.path.basename(f)}")

Found 9 PCA panels:
  curated_gene_patterns_pca_axon_guidance.svg
  curated_gene_patterns_pca_cell_adhesion.svg
  curated_gene_patterns_pca_ecm.svg
  curated_gene_patterns_pca_ion_channels.svg
  curated_gene_patterns_pca_neuromodulator_signaling.svg
  curated_gene_patterns_pca_neurotransmitter_signaling.svg
  curated_gene_patterns_pca_synaptic_transmission.svg
  curated_gene_patterns_pca_transcription.svg
  curated_gene_patterns_pca_transporters.svg


In [10]:
svg_dir

'/root/capsule/output/figures/snRNAseq'

In [11]:
svg_dir

'/root/capsule/output/figures/snRNAseq'

In [ ]:
glob.glob(os.path.join(svg_dir, "/curated_gene_patterns_pca_*.svg"))

In [16]:
# ── Parse SVGs: extract text metadata and strip text for rasterization ──
SVG_NS = "http://www.w3.org/2000/svg"

def parse_svg_panel(svg_path):
    """Parse an SVG, extract text metadata, and return a text-stripped version as bytes."""
    tree = etree.parse(svg_path)
    root = tree.getroot()

    # Collect all text elements with their content and style info
    texts = []
    for txt_el in root.iter(f"{{{SVG_NS}}}text"):
        content = txt_el.text or ""
        style = txt_el.get("style", "")
        x = txt_el.get("x", "0")
        y = txt_el.get("y", "0")
        # Extract font-size from style
        m = re.search(r"font-size:\s*([\d.]+)", style)
        font_size = float(m.group(1)) if m else 0
        texts.append({
            "content": content.strip(),
            "x": float(x), "y": float(y),
            "font_size": font_size,
            "style": style
        })

    # Identify elements by role
    gene_names = []
    class_label = None
    colorbar_info = []

    for t in texts:
        if t["font_size"] >= 11 and t["content"]:
            # 12px text: could be gene name or class label
            # Class label is typically the one at the very top (smallest y)
            pass  # classify below
        elif t["content"] == "CPM":
            colorbar_info.append(t)
        elif t["content"]:
            colorbar_info.append(t)

    # Among the 12px texts, the one with lowest y is the class title
    big_texts = [t for t in texts if t["font_size"] >= 11 and t["content"]]
    if big_texts:
        big_texts_sorted = sorted(big_texts, key=lambda t: t["y"])
        class_label = big_texts_sorted[0]["content"]  # topmost = class title
        gene_names = [t["content"] for t in big_texts_sorted[1:]]  # rest = gene names

    # Now strip ALL text from a copy of the SVG
    root_copy = copy.deepcopy(root)
    for txt_el in list(root_copy.iter(f"{{{SVG_NS}}}text")):
        txt_el.getparent().remove(txt_el)

    stripped_svg = etree.tostring(root_copy, xml_declaration=True, encoding="utf-8")
    return {
        "gene_names": gene_names,
        "class_label": class_label,
        "colorbar_texts": [t for t in texts if t["font_size"] < 11],
        "all_texts": texts,
        "stripped_svg": stripped_svg,
    }

# Parse all panels
panels = {}
for svg_path in svg_files:
    key = os.path.basename(svg_path).replace("curated_gene_patterns_pca_", "").replace(".svg", "")
    info = parse_svg_panel(svg_path)
    panels[key] = info
    print(f"{key:30s} → class: {info['class_label']:30s} genes: {info['gene_names']}")

axon_guidance                  → class: Axon guidance                  genes: ['Epha6', 'Plxna2', 'Robo1', 'Epha7', 'Plxna4', 'Sema3c', 'Slit2']
cell_adhesion                  → class: Cell adhesion                  genes: ['Csmd3', 'Lrrtm4', 'Ncam2', 'Cdh6', 'Cntn4', 'Fat3', 'Pcdh7', 'Sdk2', 'Tmem132d']
ecm                            → class: ECM                            genes: ['Chsy3', 'Col18a1', 'Col6a5', 'Hs6st3', 'Ndst3']
ion_channels                   → class: Ion channels                   genes: ['Hcn1', 'Scn9a', 'Kcnh1', 'Kctd8']
neuromodulator_signaling       → class: Neuromodulator signaling       genes: ['Trhr', 'Sctr', 'Tacr3', 'Olfr78']
neurotransmitter_signaling     → class: Neurotransmitter signaling     genes: ['Gabrg1']
synaptic_transmission          → class: Synaptic transmission          genes: ['Cadps2']
transcription                  → class: Transcription                  genes: ['Esr1', 'Nrip1', 'Tshz1', 'Tshz2']
transporters                   → class: Transp

In [17]:
# ── Check SVG dimensions and rasterize text-stripped SVGs ──
DPI = 500

def get_svg_dims(svg_bytes):
    """Get width/height from SVG viewBox or width/height attributes."""
    root = etree.fromstring(svg_bytes)
    vb = root.get("viewBox")
    if vb:
        parts = vb.split()
        return float(parts[2]), float(parts[3])
    w = root.get("width", "288pt").replace("pt", "")
    h = root.get("height", "288pt").replace("pt", "")
    return float(w), float(h)

def svg_bytes_to_image(svg_bytes, dpi=DPI):
    """Convert SVG bytes to a PIL Image at the given DPI."""
    scale = dpi / 72.0
    png_data = cairosvg.svg2png(bytestring=svg_bytes, scale=scale)
    return Image.open(io.BytesIO(png_data))

panel_images = {}
for key, info in panels.items():
    w, h = get_svg_dims(info["stripped_svg"])
    img = svg_bytes_to_image(info["stripped_svg"])
    panel_images[key] = np.array(img)
    n_genes = len(info["gene_names"])
    print(f"{key:30s}: SVG {w:.0f}×{h:.0f} pt, {n_genes} genes → {img.size[0]}×{img.size[1]} px")

print(f"\nRasterized {len(panel_images)} panels.")


KeyboardInterrupt



In [ ]:
# ── Assemble Supplemental Figure S9 ──
# Layout: 3 columns × 3 rows (9 gene-class panels)
# Each panel shows 2 genes side-by-side with colorbars (from the rasterized SVG)

# Ordered list of categories for the grid
category_order = [
    "axon_guidance",
    "cell_adhesion",
    "ecm",
    "ion_channels",
    "neuromodulator_signaling",
    "neurotransmitter_signaling",
    "synaptic_transmission",
    "transcription",
    "transporters",
]
# Keep only categories that were found
category_order = [c for c in category_order if c in panels]
n_panels = len(category_order)

# Grid dimensions
ncols = 3
nrows = int(np.ceil(n_panels / ncols))

# SVG original size is 288×288 pt → panel aspect = 1:1
panel_w_in = 3.0   # inches per panel
panel_h_in = 3.0
header_h   = 0.25  # extra space for class label

fig_w = panel_w_in * ncols + 0.3   # small side margins
fig_h = (panel_h_in + header_h) * nrows + 0.4

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(fig_w, fig_h),
    gridspec_kw={"hspace": 0.45, "wspace": 0.15}
)
axes = np.atleast_2d(axes)

for idx, cat_key in enumerate(category_order):
    row, col = divmod(idx, ncols)
    ax = axes[row, col]
    info = panels[cat_key]
    img  = panel_images[cat_key]

    ax.imshow(img, aspect="equal", interpolation="lanczos")
    ax.axis("off")

    # ── Gene class header (Helvetica 8pt) ──
    ax.set_title(
        info["class_label"],
        fontproperties=font_class,
        pad=8,
    )

    # ── Gene names (Helvetica Oblique 7pt) ──
    # Original SVG is 288pt wide; gene names sit roughly at x≈57 and x≈198 (out of 288)
    # Convert to axis fraction
    w_px = img.shape[1]
    for i, gname in enumerate(info["gene_names"]):
        # Left gene ≈ 0.20, right gene ≈ 0.69 across the panel
        x_frac = 0.20 if i == 0 else 0.69
        ax.text(
            x_frac, 1.01, gname,
            transform=ax.transAxes,
            fontproperties=font_gene,
            ha="center", va="bottom",
        )

    # ── Re-add colorbar labels (Helvetica 7pt) ──
    # Map each colorbar text from SVG pt coords → image pixel coords → axes fraction
    svg_w, svg_h = 288.0, 288.0
    for t in info["colorbar_texts"]:
        x_frac = t["x"] / svg_w
        y_frac = t["y"] / svg_h
        # Axes coordinates: SVG y increases downward, axes y increases upward in imshow
        ax.text(
            x_frac, y_frac, t["content"],
            transform=ax.transAxes,
            fontproperties=font_text,
            ha="left" if "anchor: start" in t["style"] else (
                "right" if "anchor: end" in t["style"] else "center"),
            va="center",
            fontsize=7,
        )

# Hide unused axes
for idx in range(n_panels, nrows * ncols):
    row, col = divmod(idx, ncols)
    axes[row, col].axis("off")

plt.suptitle("Supplemental Figure S9", fontproperties=font_class, fontsize=10, y=0.99)
plt.show()

In [ ]:
# ── Save final figure ──
out_dir = SNRNA_FIGURE_DIR
os.makedirs(out_dir, exist_ok=True)

# Re-create figure for saving (same code as above, wrapped in a function)
def make_figure():
    ncols = 3
    nrows = int(np.ceil(len(category_order) / ncols))
    panel_w_in, panel_h_in, header_h = 3.0, 3.0, 0.25
    fig_w = panel_w_in * ncols + 0.3
    fig_h = (panel_h_in + header_h) * nrows + 0.4

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(fig_w, fig_h),
        gridspec_kw={"hspace": 0.45, "wspace": 0.15}
    )
    axes = np.atleast_2d(axes)

    for idx, cat_key in enumerate(category_order):
        row, col = divmod(idx, ncols)
        ax = axes[row, col]
        info = panels[cat_key]
        img  = panel_images[cat_key]
        ax.imshow(img, aspect="equal", interpolation="lanczos")
        ax.axis("off")
        ax.set_title(info["class_label"], fontproperties=font_class, pad=8)
        for i, gname in enumerate(info["gene_names"]):
            x_frac = 0.20 if i == 0 else 0.69
            ax.text(x_frac, 1.01, gname, transform=ax.transAxes,
                    fontproperties=font_gene, ha="center", va="bottom")
        svg_w, svg_h = 288.0, 288.0
        for t in info["colorbar_texts"]:
            x_frac = t["x"] / svg_w
            y_frac = t["y"] / svg_h
            ha = ("left" if "anchor: start" in t["style"]
                  else ("right" if "anchor: end" in t["style"] else "center"))
            ax.text(x_frac, y_frac, t["content"], transform=ax.transAxes,
                    fontproperties=font_text, ha=ha, va="center", fontsize=7)
    for idx in range(len(category_order), nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row, col].axis("off")
    plt.suptitle("Supplemental Figure S9", fontproperties=font_class, fontsize=10, y=0.99)
    return fig

fig = make_figure()

for fmt in ("svg", "png"):
    path = os.path.join(out_dir, f"supplemental_figure_S9.{fmt}")
    fig.savefig(path, dpi=500, bbox_inches="tight",
                facecolor="white", edgecolor="none")
    size_mb = os.path.getsize(path) / 1e6
    print(f"Saved {path}  ({size_mb:.1f} MB)")

plt.close(fig)